In [1]:
import pandas as pd
import sys
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append('src')
from db_utils import get_engine
from ingestion import convert_date, read_matches, read_players, build_tournaments, build_players, build_matches

In [ ]:
df_2023 = pd.read_csv('../data/atp_matches_2023.csv')
df_2024 = pd.read_csv('../data/atp_matches_2024.csv')
df_2025 = pd.read_csv('../data/atp_matches_2025.csv')

df_2024.info()
# give the columns, non null count and data type of each column
# player_id como identificador dos jogadores

<class 'pandas.DataFrame'>
RangeIndex: 3076 entries, 0 to 3075
Data columns (total 49 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   tourney_id          3076 non-null   str    
 1   tourney_name        3076 non-null   str    
 2   surface             3076 non-null   str    
 3   draw_size           3076 non-null   int64  
 4   tourney_level       3076 non-null   str    
 5   tourney_date        3076 non-null   int64  
 6   match_num           3076 non-null   int64  
 7   winner_id           3076 non-null   int64  
 8   winner_seed         1294 non-null   float64
 9   winner_entry        477 non-null    str    
 10  winner_name         3076 non-null   str    
 11  winner_hand         3076 non-null   str    
 12  winner_ht           3057 non-null   float64
 13  winner_ioc          3076 non-null   str    
 14  winner_age          3075 non-null   float64
 15  loser_id            3076 non-null   int64  
 16  loser_seed       

In [ ]:
# o isna().sum() devolve um objeto do tipo series. A partir daí consigo filtrar

series = df_2023.isna().sum()
series = series[series > 500]
print(series)

# winner_seed     1736
# winner_entry    2513
# loser_seed      2217
# loser_entry     2291

winner_seed     1736
winner_entry    2513
loser_seed      2217
loser_entry     2291
dtype: int64


In [34]:
# A maneira que consegui para ver todas as colunas. As colunas estavam a ser ocultas, por serem muitas

df_2024.head().T

#surface
#winner_id - uma vez que vou ter uma tabela de jogadores, isto é suficiente
#winner_hand
#winner_ht
#winner_rank
#loser_id - uma vez que vou ter uma tabela de jogadores, isto é suficiente
#loser_hand
#loser_rank
#score

,0,1,2,3,4
tourney_id,2024-0339,2024-0339,2024-0339,2024-0339,2024-0339
tourney_name,Brisbane,Brisbane,Brisbane,Brisbane,Brisbane
surface,Hard,Hard,Hard,Hard,Hard
draw_size,32,32,32,32,32
tourney_level,A,A,A,A,A
tourney_date,20240101,20240101,20240101,20240101,20240101
match_num,300,299,298,297,296
winner_id,105777,208029,105777,208029,126128
winner_seed,2.0,1.0,2.0,1.0,NaN
winner_entry,NaN,NaN,NaN,NaN,NaN


In [58]:
df_all = pd.concat([df_2023, df_2024, df_2025]).sort_values(by=['tourney_date', 'match_num']).reset_index(drop=True)

df_all['UncompletedMatch'] = df_all['score'].str.contains('W/O') | df_all['score'].str.contains('RET')

In [ ]:
df_all = df_all[['tourney_id', 'tourney_name', 'tourney_date', 'tourney_level', 'match_num', 'surface', 'round', 'winner_id', 'winner_rank', 'loser_id', 'loser_rank', 'score', 'UncompletedMatch']]

df_all.head()

,surface,winner_id,winner_hand,winner_rank,loser_id,loser_hand,loser_rank,score
0,Hard,105777,R,14.0,208029,R,8.0,7-6(5) 6-4
1,Hard,208029,R,8.0,126128,R,39.0,6-4 7-6(0)
2,Hard,105777,R,14.0,111442,R,55.0,6-3 7-5
3,Hard,208029,R,8.0,105902,R,116.0,6-2 7-6(6)
4,Hard,126128,R,39.0,208286,R,44.0,7-6(4) 6-2


In [88]:
df_all['is_walkover'] = df_all['score'].str.contains('W/O')
df_all['is_retirement'] = df_all['score'].str.contains('RET')

df_all.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,UncompletedMatch,is_walkover,is_retirement
0,2023-9900,United Cup,Hard,18,A,20230102,207,127339,NaN,NaN,...,11.0,3.0,4.0,144.0,381.0,75.0,695.0,False,False,False
1,2023-9900,United Cup,Hard,18,A,20230102,208,106432,8.0,NaN,...,11.0,5.0,9.0,26.0,1685.0,30.0,1395.0,False,False,False
2,2023-9900,United Cup,Hard,18,A,20230102,211,105173,NaN,NaN,...,6.0,4.0,10.0,46.0,946.0,75.0,695.0,False,False,False
3,2023-9900,United Cup,Hard,18,A,20230102,212,126239,6.0,NaN,...,9.0,4.0,7.0,44.0,952.0,30.0,1395.0,False,False,False
4,2023-9900,United Cup,Hard,18,A,20230102,215,127339,NaN,NaN,...,17.0,3.0,5.0,144.0,381.0,46.0,946.0,False,False,False


In [90]:
df_players = pd.read_csv('../data/atp_players.csv', dtype={'wikidata_id': str})

df_players.head()

,player_id,name_first,name_last,hand,dob,ioc,height,wikidata_id
0,100001,Gardnar,Mulloy,R,19131122.0,USA,185.0,Q54544
1,100002,Pancho,Segura,R,19210620.0,ECU,168.0,Q54581
2,100003,Frank,Sedgman,R,19271002.0,AUS,180.0,Q962049
3,100004,Giuseppe,Merlo,R,19271011.0,ITA,NaN,Q1258752
4,100005,Richard,Gonzalez,R,19280509.0,USA,188.0,Q53554


In [97]:
df_players.head()

,player_id,name_first,name_last,hand,dob,ioc,height,wikidata_id
0,100001,Gardnar,Mulloy,R,19131122.0,USA,185.0,Q54544
1,100002,Pancho,Segura,R,19210620.0,ECU,168.0,Q54581
2,100003,Frank,Sedgman,R,19271002.0,AUS,180.0,Q962049
3,100004,Giuseppe,Merlo,R,19271011.0,ITA,NaN,Q1258752
4,100005,Richard,Gonzalez,R,19280509.0,USA,188.0,Q53554


In [101]:
isEmpty_tourneyDate = df_all[df_all['tourney_date'].isna() == True]
print(isEmpty_tourneyDate)


Empty DataFrame
Columns: [tourney_id, tourney_name, surface, draw_size, tourney_level, tourney_date, match_num, winner_id, winner_seed, winner_entry, winner_name, winner_hand, winner_ht, winner_ioc, winner_age, loser_id, loser_seed, loser_entry, loser_name, loser_hand, loser_ht, loser_ioc, loser_age, score, best_of, round, minutes, w_ace, w_df, w_svpt, w_1stIn, w_1stWon, w_2ndWon, w_SvGms, w_bpSaved, w_bpFaced, l_ace, l_df, l_svpt, l_1stIn, l_1stWon, l_2ndWon, l_SvGms, l_bpSaved, l_bpFaced, winner_rank, winner_rank_points, loser_rank, loser_rank_points, UncompletedMatch, is_walkover, is_retirement]
Index: []

[0 rows x 52 columns]


In [100]:
isEmpty_dob = df_players[df_players['dob'].isna() == True]
print(isEmpty_dob)

       player_id name_first      name_last hand  dob  ioc  height wikidata_id
6419      106434          A         Benson    R  NaN  AUS     NaN         NaN
6420      106435          A        Escofet    R  NaN  ESP     NaN         NaN
6421      106436          A           Hall    R  NaN  AUS     NaN         NaN
6422      106437          A         Noffat    R  NaN  GRE     NaN         NaN
6423      106438          A         Uehara    R  NaN  JPN     NaN         NaN
...          ...        ...            ...  ...  ...  ...     ...         ...
66806     214524      Ramon         Mateos    U  NaN  ECU     NaN         NaN
66808     214526     Sergio  Castro Loedel    U  NaN  ECU     NaN         NaN
66838     214556      Jakub        Smejcky    U  NaN  CZE     NaN         NaN
66873     214591     Avetik      Ghazaryan    U  NaN  ARM     NaN         NaN
66880     214598       Snir          Morag    U  NaN  ISR     NaN         NaN

[18639 rows x 8 columns]


In [106]:
converted_dates = convert_date(df_players['dob'])

print(converted_dates)

0       1913-11-22
1       1921-06-20
2       1927-10-02
3       1927-10-11
4       1928-05-09
           ...    
66907   2009-02-08
66908   2003-03-21
66909   2007-03-29
66910   2009-09-16
66911   2007-01-03
Name: dob, Length: 66912, dtype: datetime64[us]


In [107]:
before = df_players['dob'].isna().sum()
after = convert_date(df_players['dob']).isna().sum()
print(before, after)

18639 18741


In [131]:
converted = convert_date(df_players['dob'])
lost = df_players['dob'].notna() & converted.isna()
df_lost = df_players.loc[lost, ['player_id', 'name_first', 'name_last', 'dob']]

df_playerLost = (df_all['winner_id'].isin(df_lost['player_id']) | df_all['loser_id'].isin(df_lost['player_id'])).sum()
print(df_playerLost)
# | df_all['loser_id'].isin(df_lost['player_id'])
# print(df_playerLost.sum())

0


In [120]:
df_lost = df_players.loc[lost, ['player_id', 'name_first', 'name_last', 'dob']]
df_lost.head()

matches = read_matches([2023, 2024, 2025])
print(matches)

There's an error while extracting matches data:  [Errno 2] No such file or directory: 'data/atp_matches_2023.csv'
None


In [ ]:
df_lost = df_all['winner_id', 'loser_id']
df_lost.head()

0    127339
1    106432
2    105173
3    126239
4    127339
Name: winner_id, dtype: int64

In [3]:
matches = read_matches([2023, 2024, 2025])
tourneys = build_tournaments(matches)
print(len(tourneys))

445


In [4]:
players_raw = read_players()
players = build_players(players_raw)

# 1. Same number of rows (no player dropped)
print(len(players_raw), len(players))

# 2. dob is now a date
print(players.dtypes)

# 3. No missing last names (schema says NOT NULL)
print(players['name_last'].isna().sum())

# 4. Missing dob before vs after (difference should be the 102 unparseable dates)
print(players_raw['dob'].isna().sum(), players['dob'].isna().sum())

66912 66912
player_id              int64
name_first               str
name_last                str
hand                     str
dob           datetime64[us]
ioc                      str
height               float64
dtype: object
48
18639 18741


In [5]:
matches_raw = read_matches([2023, 2024, 2025])
matches = build_matches(matches_raw)
print(len(matches_raw), len(matches))   # equal: no match dropped
print(matches['is_wo'].sum(), matches['is_retirement'].sum())   # how many of each
print(matches.columns.tolist())   # compare with TennisMatch in schema.sql

9006 9006
62 266
['tourney_id', 'match_num', 'round', 'winner_id', 'loser_id', 'winner_rank', 'loser_rank', 'score', 'is_wo', 'is_retirement']
